# MethylSeg sweep figure browser

This notebook scans the archived sweep outputs under `results/00_methylseg_development/random_parallel_sweep/`, ranks configs from the summary tables, and then renders each saved figure inline so you can quickly scroll through every config.

It supports the archived sweep outputs:

- interactive `.html` figures rendered in an iframe
- static image outputs such as `.png`, `.jpg`, `.jpeg`, and `.svg`


In [ ]:
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, Markdown, display

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "repo_paths.py").exists() and (candidate / "analysis").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the MethylSegPaper repository root from the current working directory.")


PROJECT_ROOT = resolve_project_root()
SWEEP_ROOT = PROJECT_ROOT / "results" / "00_methylseg_development" / "random_parallel_sweep"
if not (SWEEP_ROOT / "sweep_results.tsv").exists():
    raise FileNotFoundError(f"Archived sweep results were not found: {SWEEP_ROOT}")
MANIFEST_PATH = SWEEP_ROOT / "sweep_manifest.tsv"
RESULTS_PATH = SWEEP_ROOT / "sweep_results.tsv"
FIGURE_SUFFIXES = {".html", ".png", ".jpg", ".jpeg", ".svg"}
HTML_HEIGHT = 540
IMAGE_WIDTH = 1100


def load_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected table: {path}")
    return pd.read_csv(path, sep="\t")


def iter_config_result_roots(sweep_root: Path):
    for config_dir in sorted(p for p in sweep_root.iterdir() if p.is_dir() and p.name.startswith("random_")):
        results_dir = config_dir / "results"
        if results_dir.exists():
            yield config_dir.name, results_dir


def collect_figure_records(sweep_root: Path) -> pd.DataFrame:
    rows = []
    for config_name, results_dir in iter_config_result_roots(sweep_root):
        for figure_path in sorted(
            path for path in results_dir.rglob("*") if path.is_file() and path.suffix.lower() in FIGURE_SUFFIXES
        ):
            rel_path = figure_path.relative_to(results_dir)
            rows.append(
                {
                    "config_name": config_name,
                    "figure_type": figure_path.suffix.lower().lstrip("."),
                    "section": rel_path.parts[0] if rel_path.parts else "results",
                    "label": rel_path.stem,
                    "relative_path": str(rel_path),
                    "path": figure_path,
                }
            )
    return pd.DataFrame(rows)


def render_html_figure(path: Path, height: int = HTML_HEIGHT):
    srcdoc = escape(path.read_text(encoding="utf-8", errors="ignore"), quote=True)
    return HTML(
        f'<iframe srcdoc="{srcdoc}" width="100%" height="{height}" '
        'style="border: 1px solid #d0d7de; border-radius: 6px; background: white;"></iframe>'
    )


def render_image_figure(path: Path, width: int = IMAGE_WIDTH):
    return Image(filename=str(path), width=width)


def render_figure(path: Path):
    if path.suffix.lower() == ".html":
        return render_html_figure(path)
    return render_image_figure(path)


def collect_known_samples(manifest_df: pd.DataFrame) -> list[str]:
    sample_names = set()
    if "samples" not in manifest_df.columns:
        return []
    for value in manifest_df["samples"].dropna():
        for sample_name in str(value).split(","):
            sample_name = sample_name.strip()
            if sample_name:
                sample_names.add(sample_name)
    return sorted(sample_names)


def filter_figures_by_samples(figures_df: pd.DataFrame, sample_names: list[str] | None, known_samples: list[str]) -> pd.DataFrame:
    if not sample_names:
        return figures_df.copy()

    sample_names = [sample_name.strip() for sample_name in sample_names if str(sample_name).strip()]
    if not sample_names:
        return figures_df.copy()

    requested_sample_set = set(sample_names)
    unknown_samples = sorted(requested_sample_set - set(known_samples))
    if unknown_samples:
        print(f"Warning: requested samples were not found in the sweep manifest: {unknown_samples}")

    def row_matches(row) -> bool:
        rel_path = row.relative_path
        matching_known_samples = [sample for sample in known_samples if sample in rel_path]
        if not matching_known_samples:
            return True
        return any(sample in requested_sample_set for sample in matching_known_samples)

    return figures_df[figures_df.apply(row_matches, axis=1)].reset_index(drop=True)


def normalize_config_names(config_names: list[str] | None, available_config_names: list[str]) -> list[str] | None:
    if config_names is None:
        return None

    available_lookup = {name.lower(): name for name in available_config_names}
    normalized = []
    unknown = []

    for raw_name in config_names:
        name = str(raw_name).strip()
        if not name:
            continue
        lowered = name.lower()
        candidate_names = [lowered]
        if lowered.startswith("random_"):
            suffix = lowered.split("_", 1)[1]
            if suffix.isdigit():
                candidate_names.append(f"random_{int(suffix):02d}")

        matched_name = None
        for candidate in candidate_names:
            if candidate in available_lookup:
                matched_name = available_lookup[candidate]
                break

        if matched_name is None:
            unknown.append(name)
        elif matched_name not in normalized:
            normalized.append(matched_name)

    if unknown:
        print(f"Warning: requested configs were not found: {unknown}")

    return normalized


def load_per_config_table(config_name: str, relative_path: str) -> pd.DataFrame:
    table_path = SWEEP_ROOT / config_name / "results" / relative_path
    if not table_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(table_path, sep="\t")
    df.insert(0, "config_name", config_name)
    return df


def combine_config_tables(config_names: list[str], relative_path: str) -> pd.DataFrame:
    frames = [load_per_config_table(config_name, relative_path) for config_name in config_names]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def rank_synthetic_configs(synthetic_df: pd.DataFrame) -> pd.DataFrame:
    if synthetic_df.empty:
        return pd.DataFrame()

    ranked = (
        synthetic_df.assign(
            f1_like=lambda df: (2 * df["bp_precision"] * df["bp_recall"]) /
            (df["bp_precision"] + df["bp_recall"]).replace(0, pd.NA)
        )
        .groupby(["config_name", "platform"], as_index=False)
        .agg(
            n_samples=("sample", "nunique"),
            mean_bp_recall=("bp_recall", "mean"),
            mean_bp_precision=("bp_precision", "mean"),
            mean_f1_like=("f1_like", "mean"),
        )
        .sort_values(["platform", "mean_f1_like", "mean_bp_recall", "mean_bp_precision"], ascending=[True, False, False, False])
        .reset_index(drop=True)
    )
    ranked["rank"] = ranked.groupby("platform").cumcount() + 1
    return ranked[["platform", "rank", "config_name", "n_samples", "mean_f1_like", "mean_bp_recall", "mean_bp_precision"]]


def rank_chromatin_configs(chromatin_df: pd.DataFrame) -> pd.DataFrame:
    if chromatin_df.empty:
        return pd.DataFrame()

    ranked = (
        chromatin_df.groupby(["config_name", "platform"], as_index=False)
        .agg(
            n_samples=("sample", "nunique"),
            mean_signal_ratio=("signal_ratio", "mean"),
            mean_peak_ratio=("peak_ratio", "mean"),
            mean_peak_overlap_fraction=("peak_overlap_fraction", "mean"),
        )
        .assign(composite_score=lambda df: (df["mean_signal_ratio"] + df["mean_peak_ratio"] + df["mean_peak_overlap_fraction"]) / 3.0)
        .sort_values(["platform", "composite_score", "mean_signal_ratio", "mean_peak_ratio"], ascending=[True, False, False, False])
        .reset_index(drop=True)
    )
    ranked["rank"] = ranked.groupby("platform").cumcount() + 1
    return ranked[["platform", "rank", "config_name", "n_samples", "composite_score", "mean_signal_ratio", "mean_peak_ratio", "mean_peak_overlap_fraction"]]


def rank_lad_configs(lad_df: pd.DataFrame) -> pd.DataFrame:
    if lad_df.empty:
        return pd.DataFrame()

    ranked = (
        lad_df.groupby(["config_name", "platform"], as_index=False)
        .agg(
            n_samples=("sample", "nunique"),
            mean_pct_lad_end_overlap=("pct_regions_with_lad_at_either_end", "mean"),
            mean_n_regions=("n_regions", "mean"),
        )
        .sort_values(["platform", "mean_pct_lad_end_overlap", "mean_n_regions"], ascending=[True, False, False])
        .reset_index(drop=True)
    )
    ranked["rank"] = ranked.groupby("platform").cumcount() + 1
    return ranked[["platform", "rank", "config_name", "n_samples", "mean_pct_lad_end_overlap", "mean_n_regions"]]


manifest_df = load_table(MANIFEST_PATH)
results_df = load_table(RESULTS_PATH)
figures_df = collect_figure_records(SWEEP_ROOT)
known_samples = collect_known_samples(manifest_df)

display(Markdown(f"Loaded `{len(results_df)}` sweep configs and `{len(figures_df)}` renderable figure files from `{SWEEP_ROOT}`."))
display(Markdown(f"Known primary samples: `{', '.join(known_samples) if known_samples else 'none detected'}`"))


In [ ]:
summary_columns = [
    "config_name",
    "status",
    "n_primary_samples",
    "n_chromatin_rows",
    "n_lad_rows",
    "n_synthetic_runs",
    "n_synthetic_summary_rows",
    "error",
]

display(results_df[summary_columns].sort_values("config_name").reset_index(drop=True))

figure_counts_df = (
    figures_df.groupby(["config_name", "figure_type"]).size().unstack(fill_value=0).reset_index()
    if not figures_df.empty
    else pd.DataFrame(columns=["config_name"])
)
display(figure_counts_df)

display(figures_df[["config_name", "section", "figure_type", "relative_path"]].reset_index(drop=True))


In [ ]:
# Optional filters:
# - Leave CONFIG_NAMES as None to render every config.
# - Config names are normalized, so values like "random_4" map to "random_04".
# - Leave SAMPLE_NAMES as None to render all samples.
# - SAMPLE_NAMES only filters sample-specific figures; global figures stay visible.
CONFIG_NAMES = None
SAMPLE_NAMES = None

CONFIG_NAMES = normalize_config_names(CONFIG_NAMES, sorted(figures_df["config_name"].unique()))
render_df = figures_df.sort_values(["config_name", "relative_path"]).reset_index(drop=True)
if CONFIG_NAMES is not None:
    render_df = render_df[render_df["config_name"].isin(CONFIG_NAMES)].reset_index(drop=True)
render_df = filter_figures_by_samples(render_df, SAMPLE_NAMES, known_samples)

status_lookup = results_df.set_index("config_name")["status"].to_dict()
figure_count_lookup = render_df.groupby("config_name").size().to_dict()
sample_label = ", ".join(SAMPLE_NAMES) if SAMPLE_NAMES else "all"
config_label = ", ".join(CONFIG_NAMES) if CONFIG_NAMES else "all"
display(Markdown(f"Rendering config filter: `{config_label}`"))
display(Markdown(f"Rendering sample filter: `{sample_label}`"))


In [ ]:
# Rank configs best-to-worst within each platform using the selected CONFIG_NAMES subset.
# Synthetic ranking uses mean F1-like score from bp precision and recall.
# Functional analysis ranking uses a composite of signal ratio, peak ratio, and peak overlap fraction.
# LAD ranking uses the mean percent of regions with LAD at either end.

selected_config_names = CONFIG_NAMES or sorted(results_df["config_name"].dropna().unique())

synthetic_rank_input = combine_config_tables(
    selected_config_names,
    "synthetic/analysis/tables/synthetic_methylseg_summary.tsv",
)
if not synthetic_rank_input.empty and "tool" in synthetic_rank_input.columns:
    synthetic_rank_input = synthetic_rank_input.copy()
    synthetic_rank_input["platform"] = synthetic_rank_input["tool"].map({"methylseg": "wgbs", "methylseg_hm450k": "hm450k"})

chromatin_rank_input = combine_config_tables(
    selected_config_names,
    "functional_analysis/chromatin_fallback/tables/chromatin_methylseg_metrics.tsv",
)
lad_rank_input = combine_config_tables(
    selected_config_names,
    "functional_analysis/lad_fallback/tables/lad_metrics.tsv",
)

synthetic_ranking_df = rank_synthetic_configs(synthetic_rank_input)
chromatin_ranking_df = rank_chromatin_configs(chromatin_rank_input)
lad_ranking_df = rank_lad_configs(lad_rank_input)

display(Markdown("## Config rankings"))
display(Markdown(f"Ranked configs: `{', '.join(selected_config_names)}`"))

for platform in ["wgbs", "hm450k"]:
    display(Markdown(f"### {platform.upper()}"))
    display(Markdown("Synthetic"))
    display(
        synthetic_ranking_df.loc[synthetic_ranking_df["platform"] == platform, [
            "rank", "config_name", "n_samples", "mean_f1_like", "mean_bp_recall", "mean_bp_precision"
        ]].reset_index(drop=True)
    )
    display(Markdown("Functional analysis"))
    display(
        chromatin_ranking_df.loc[chromatin_ranking_df["platform"] == platform, [
            "rank", "config_name", "n_samples", "composite_score", "mean_signal_ratio", "mean_peak_ratio", "mean_peak_overlap_fraction"
        ]].reset_index(drop=True)
    )
    display(Markdown("LAD"))
    display(
        lad_ranking_df.loc[lad_ranking_df["platform"] == platform, [
            "rank", "config_name", "n_samples", "mean_pct_lad_end_overlap", "mean_n_regions"
        ]].reset_index(drop=True)
    )


In [ ]:
for config_name, config_figures in render_df.groupby("config_name", sort=True):
    status = status_lookup.get(config_name, "unknown")
    n_figures = figure_count_lookup.get(config_name, len(config_figures))
    display(Markdown(f"---\n## {config_name}\n\nStatus: `{status}`  \nFigures found: `{n_figures}`"))
    for row in config_figures.itertuples(index=False):
        display(Markdown(f"### `{row.relative_path}`"))
        display(render_figure(row.path))
